# InferenceOverviewSimExtended_plots

Plot script for the §4 cross-fitting / DML simulation figures.

Reads the saved simulation results from
`Data/InferenceOverviewSimExtended.csv` and writes the three slide figures
to `Slides/figures/`:

- `Slide_26.png` — nonlinear DGP, Oracle vs. Full-sample DNN
  (overfitting-bias panel)
- `Slide_30.png` — nonlinear DGP, Oracle vs. DML(DNN) vs. Full-sample DNN
  (cross-fitting fixes the bias)
- `Slide_34.png` — 2-panel "match learner to DGP":
  left = linear DGP, right = nonlinear DGP

Aesthetic conventions (from session 18):
- No panel titles
- Common x-axis (0.10, 0.90) and common bin edges across every panel
- Legend below the x-axis as a horizontal strip, no frame
- Vertical dashed line at the true alpha_0 = 0.5 for orientation

The CSV is generated by `InferenceOverviewSimExtended.py` /
`InferenceOverviewSimExtended.ipynb`; rerunning the simulation is not
required to regenerate the figures.

In [ ]:
from __future__ import annotations
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE      = Path(os.getcwd()).resolve()
REPO      = HERE.parent if HERE.name == "Python code" else HERE
DATA_CSV  = (REPO / "Data" / "InferenceOverviewSimExtended.csv").resolve()
PLOTDIR   = (REPO / "Slides" / "figures").resolve()

ALPHA0    = 0.5

# Common x-axis and bins (40 bins, width 0.02; alpha_0 lands on a bin edge).
XLIM      = (0.10, 0.90)
BINS      = np.arange(XLIM[0], XLIM[1] + 1e-9, 0.02)

# Palette: Oracle green, DML(Lasso) orange, DML(DNN) red, full-sample DNN steel blue,
# truth line black so it does not clash with DML(DNN)'s red.
C_ORACLE  = "#3a8c3a"
C_LASSO   = "#d97a00"
C_DNN     = "#b03030"
C_FULL    = "#2a4d8f"
C_TRUTH   = "black"

ALPHA_FILL = 0.62

plt.rcParams.update({
    "font.size":        13,
    "axes.labelsize":   14,
    "axes.titlesize":   14,
    "xtick.labelsize":  12,
    "ytick.labelsize":  12,
    "legend.fontsize":  12,
})
print(f"DATA_CSV = {DATA_CSV}")
print(f"PLOTDIR  = {PLOTDIR}")

In [ ]:
# Helpers
def _draw_hist(ax, values, color, label):
    ax.hist(values, bins=BINS, color=color, alpha=ALPHA_FILL,
            label=label, edgecolor="white", linewidth=0.3)


def _truth_line(ax):
    ax.axvline(ALPHA0, color=C_TRUTH, linestyle="--", linewidth=1.2)


def _style_axes(ax):
    ax.set_xlim(*XLIM)
    ax.set_xlabel(r"$\hat\alpha$")
    ax.set_ylabel("Count")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _legend_below(fig, handles, labels, n_panels=1, y=-0.02):
    fig.legend(handles, labels,
               loc="lower center",
               bbox_to_anchor=(0.5, y),
               ncol=len(labels),
               frameon=False,
               handlelength=1.6,
               columnspacing=2.0)

In [ ]:
# Load CSV. The canonical 1000-rep file lives at
# Data/InferenceOverviewSimExtended.csv. If it is missing, run
# InferenceOverviewSimExtended.ipynb (or .py) with n_rep >= 50.
if not DATA_CSV.exists():
    raise FileNotFoundError(
        f"Expected simulation results at {DATA_CSV}. "
        f"Run InferenceOverviewSimExtended.py to regenerate."
    )
df = pd.read_csv(DATA_CSV)
print(f"Loaded {len(df)} reps from {DATA_CSV}")
df.describe()

In [ ]:
# Slide 26 -- nonlinear DGP, Oracle vs. Full-sample DNN.
def plot_slide_26(df, savepath=None):
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    _draw_hist(ax, df["nl_oracle"],   C_ORACLE, "Oracle")
    _draw_hist(ax, df["nl_full_dnn"], C_FULL,   "Full sample (DNN)")
    _truth_line(ax)
    _style_axes(ax)
    handles, labels = ax.get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    _legend_below(fig, handles, labels, n_panels=1, y=0.00)
    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")
    return fig

PLOTDIR.mkdir(parents=True, exist_ok=True)
fig26 = plot_slide_26(df, PLOTDIR / "Slide_26.png")
plt.show()

In [ ]:
# Slide 30 -- nonlinear DGP, Oracle vs. DML(DNN) vs. Full-sample DNN.
def plot_slide_30(df, savepath=None):
    fig, ax = plt.subplots(figsize=(8.0, 4.6))
    _draw_hist(ax, df["nl_oracle"],   C_ORACLE, "Oracle")
    _draw_hist(ax, df["nl_dml_dnn"],  C_DNN,    "DML (DNN)")
    _draw_hist(ax, df["nl_full_dnn"], C_FULL,   "Full sample (DNN)")
    _truth_line(ax)
    _style_axes(ax)
    handles, labels = ax.get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    _legend_below(fig, handles, labels, n_panels=1, y=0.00)
    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")
    return fig

fig30 = plot_slide_30(df, PLOTDIR / "Slide_30.png")
plt.show()

In [ ]:
# Slide 34 -- 2-panel "match learner to DGP".
def plot_slide_34(df, savepath=None):
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6),
                             sharey=True, sharex=True)
    for ax, prefix in ((axes[0], "lin"), (axes[1], "nl")):
        _draw_hist(ax, df[f"{prefix}_oracle"],  C_ORACLE, "Oracle")
        _draw_hist(ax, df[f"{prefix}_dml_las"], C_LASSO,  "DML (CV-Lasso)")
        _draw_hist(ax, df[f"{prefix}_dml_dnn"], C_DNN,    "DML (DNN)")
        _truth_line(ax)
        _style_axes(ax)
    axes[1].set_ylabel("")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    _legend_below(fig, handles, labels, n_panels=2, y=0.00)
    if savepath is not None:
        fig.savefig(savepath, dpi=200, bbox_inches="tight")
    return fig

fig34 = plot_slide_34(df, PLOTDIR / "Slide_34.png")
plt.show()
print(f"Saved figures -> {PLOTDIR}")